# Notebook Goal

This notebook builds and tests a reusable inference pipeline for the Credit Card Fraud Detection project.

Notebook `17_final_model_training.ipynb` already trained and saved the final validated fraud model and its supporting artifacts. Notebook `18_inference_pipeline.ipynb` will load those saved artifacts and use them to make predictions.

No model training happens in this notebook. No threshold tuning happens in this notebook. The selected features and saved decision policy from notebook 17 are reused as-is.

The purpose of this notebook is to simulate how the fraud model will work in production: load the saved model, prepare input data in the expected format, generate a fraud probability, and convert that result into a reusable prediction workflow.

This notebook also prepares the project for the next FastAPI `/predict` endpoint by turning the saved model artifacts into a clear, testable inference path.


# Load Saved Artifacts

This section loads the final validated model and the supporting JSON artifacts created in notebook `17_final_model_training.ipynb`.

The goal here is only to verify that the saved inference assets exist and can be loaded correctly before later steps use them for prediction.


In [10]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier


In [11]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

FINAL_MODEL_PATH = ARTIFACTS_DIR / "final_validated_fraud_model.joblib"
FINAL_FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "final_feature_columns.json"
FINAL_DECISION_POLICY_PATH = ARTIFACTS_DIR / "final_decision_policy.json"
FINAL_MODEL_METADATA_PATH = ARTIFACTS_DIR / "final_model_metadata.json"
FINAL_MODEL_METRICS_PATH = ARTIFACTS_DIR / "final_model_metrics.json"

artifact_paths = {
    "final_validated_model": FINAL_MODEL_PATH,
    "final_feature_columns": FINAL_FEATURE_COLUMNS_PATH,
    "final_decision_policy": FINAL_DECISION_POLICY_PATH,
    "final_model_metadata": FINAL_MODEL_METADATA_PATH,
    "final_model_metrics": FINAL_MODEL_METRICS_PATH,
}

for artifact_name, artifact_path in artifact_paths.items():
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing required artifact: {artifact_path}")

    print(f"Found {artifact_name}: {artifact_path}")


Found final_validated_model: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Found final_feature_columns: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Found final_decision_policy: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_decision_policy.json
Found final_model_metadata: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metadata.json
Found final_model_metrics: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metrics.json


In [12]:
final_model = joblib.load(FINAL_MODEL_PATH)
print(f"Loaded final validated model successfully: {FINAL_MODEL_PATH}")

with open(FINAL_FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
    final_feature_columns = json.load(f)
print(f"Loaded final feature columns successfully: {FINAL_FEATURE_COLUMNS_PATH}")

with open(FINAL_DECISION_POLICY_PATH, "r", encoding="utf-8") as f:
    final_decision_policy = json.load(f)
print(f"Loaded final decision policy successfully: {FINAL_DECISION_POLICY_PATH}")

with open(FINAL_MODEL_METADATA_PATH, "r", encoding="utf-8") as f:
    final_model_metadata = json.load(f)
print(f"Loaded final model metadata successfully: {FINAL_MODEL_METADATA_PATH}")

with open(FINAL_MODEL_METRICS_PATH, "r", encoding="utf-8") as f:
    final_model_metrics = json.load(f)
print(f"Loaded final model metrics successfully: {FINAL_MODEL_METRICS_PATH}")


Loaded final validated model successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Loaded final feature columns successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Loaded final decision policy successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_decision_policy.json
Loaded final model metadata successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metadata.json
Loaded final model metrics successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metrics.json


# Validate Artifact Consistency

This section checks that the loaded model and saved artifacts are aligned before they are used for inference.

If anything important is missing or mismatched, the notebook stops early with a clear validation error.


In [13]:
if not isinstance(final_feature_columns, list) or not final_feature_columns:
    raise ValueError("Artifact validation failed: final_feature_columns must be a non-empty list.")

if not isinstance(final_model, RandomForestClassifier):
    raise ValueError(
        "Artifact validation failed: loaded model must be a RandomForestClassifier. "
        f"Found {type(final_model).__name__}."
    )

if "review_threshold" not in final_decision_policy:
    raise ValueError(
        "Artifact validation failed: final_decision_policy is missing review_threshold."
    )

if "block_threshold" not in final_decision_policy:
    raise ValueError(
        "Artifact validation failed: final_decision_policy is missing block_threshold."
    )

review_threshold = final_decision_policy["review_threshold"]
block_threshold = final_decision_policy["block_threshold"]

if review_threshold >= block_threshold:
    raise ValueError(
        "Artifact validation failed: review_threshold must be less than block_threshold. "
        f"Found review_threshold={review_threshold} and block_threshold={block_threshold}."
    )

actual_feature_count = len(final_feature_columns)

if hasattr(final_model, "n_features_in_"):
    expected_feature_count = final_model.n_features_in_
elif "feature_count" in final_model_metadata:
    expected_feature_count = final_model_metadata["feature_count"]
else:
    raise ValueError(
        "Artifact validation failed: could not determine the model's expected feature count."
    )

if expected_feature_count != actual_feature_count:
    raise ValueError(
        "Artifact validation failed: model feature count does not match final_feature_columns. "
        f"Expected {expected_feature_count}, found {actual_feature_count}."
    )

model_version = final_model_metadata.get("model_version") or final_model_metadata.get("version")

print(f"Model type: {type(final_model).__name__}")
print(f"Expected feature count: {expected_feature_count}")
print(f"Actual feature count: {actual_feature_count}")
print(f"Review threshold: {review_threshold}")
print(f"Block threshold: {block_threshold}")
print(f"Model version: {model_version if model_version is not None else 'Not available'}")


Model type: RandomForestClassifier
Expected feature count: 13
Actual feature count: 13
Review threshold: 0.35
Block threshold: 0.5
Model version: Not available


# Create Input Validation Function

This section defines a small helper that validates one transaction before any inference step uses it.

The function checks required features, verifies numeric values, and safely ignores extra input columns.


In [14]:
def validate_transaction_input(transaction, feature_columns):
    if isinstance(transaction, pd.Series):
        transaction_data = transaction.to_dict()
    elif isinstance(transaction, dict):
        transaction_data = transaction
    else:
        raise ValueError(
            "Transaction input validation failed: transaction must be a dict or pandas Series."
        )

    if not transaction_data:
        raise ValueError(
            "Transaction input validation failed: transaction input cannot be empty."
        )

    missing_features = [
        feature_name for feature_name in feature_columns if feature_name not in transaction_data
    ]
    if missing_features:
        raise ValueError(
            "Transaction input validation failed: missing required features: "
            f"{missing_features}"
        )

    non_numeric_features = []
    for feature_name in feature_columns:
        feature_value = transaction_data[feature_name]

        if isinstance(feature_value, bool) or not isinstance(
            feature_value, (int, float, np.integer, np.floating)
        ):
            non_numeric_features.append(
                f"{feature_name}={feature_value!r} ({type(feature_value).__name__})"
            )

    if non_numeric_features:
        raise ValueError(
            "Transaction input validation failed: required features must be numeric. "
            f"Found invalid values: {non_numeric_features}"
        )

    extra_columns = [
        column_name for column_name in transaction_data if column_name not in feature_columns
    ]
    if extra_columns:
        print(f"Warning: ignoring extra input columns: {extra_columns}")

    return True


# Create Preprocessing Function

This section prepares one validated transaction in the exact feature order expected by the saved model.

The helper returns a one-row DataFrame that is ready for a later inference step.


In [15]:
def prepare_model_input(transaction, feature_columns):
    validate_transaction_input(transaction, feature_columns)

    if isinstance(transaction, pd.Series):
        transaction_data = transaction.to_dict()
    else:
        transaction_data = dict(transaction)

    ordered_transaction = {
        feature_name: transaction_data[feature_name] for feature_name in feature_columns
    }

    model_input_df = pd.DataFrame([ordered_transaction], columns=feature_columns)
    model_input_df = model_input_df.apply(pd.to_numeric)

    expected_shape = (1, len(feature_columns))
    if model_input_df.shape != expected_shape:
        raise ValueError(
            "Model input preparation failed: prepared DataFrame has the wrong shape. "
            f"Expected {expected_shape}, found {model_input_df.shape}."
        )

    return model_input_df


In [16]:
sample_transaction = {
    feature_name: float(index)
    for index, feature_name in enumerate(final_feature_columns, start=1)
}
sample_transaction["extra_input_column"] = 999.0

prepared_sample_input = prepare_model_input(sample_transaction, final_feature_columns)
print(prepared_sample_input.shape)
print(prepared_sample_input.columns.tolist())
prepared_sample_input.head()


(1, 13)
['V14_V12_interaction', 'V14', 'V17_V16_interaction', 'V12', 'V17', 'V10', 'V4', 'V16', 'V3', 'V11', 'V7', 'V18', 'log_amount']


,V14_V12_interaction,V14,V17_V16_interaction,V12,V17,V10,V4,V16,V3,V11,V7,V18,log_amount
0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0
